# Project: quantum coins
**Quantum Computing · Week 1 (Qubits and measurement) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qc/w1d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Project: quantum coins. The `ry` gate turns $|0\rangle$ toward $|1\rangle$ by an angle $\theta$:
$R_Y(\theta)|0\rangle = \cos\frac{\theta}{2}|0\rangle + \sin\frac{\theta}{2}|1\rangle$, so $P(1) = \sin^2\frac{\theta}{2}$.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler(seed=5)

def coin(theta):
    """A one-qubit circuit that lands on 1 with probability sin^2(theta / 2)."""
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.measure_all()
    return qc

def theta_for(p):
    """The angle that gives P(1) = p: invert p = sin^2(theta / 2)."""
    return 2 * np.arcsin(np.sqrt(p))

theta = theta_for(0.3)
counts = sampler.run([coin(theta)], shots=4000).result()[0].data.meas.get_counts()
print(f"theta = {np.degrees(theta):.2f} degrees")
print(f"4000 flips: {counts}  ->  P(1) = {counts.get('1', 0) / 4000:.3f}")
print(coin(theta).draw("text"))

## The whole family of coins
Simulated points (500 shots each) sit on the curve $\sin^2(\theta/2)$.

In [ ]:
import matplotlib.pyplot as plt

angles = np.radians(np.arange(0, 181, 15))
circuits = [coin(a) for a in angles]
results = sampler.run(circuits, shots=500).result()
measured = [r.data.meas.get_counts().get("1", 0) / 500 for r in results]

grid = np.linspace(0, np.pi, 200)
plt.figure(figsize=(6.2, 3.4))
plt.plot(np.degrees(grid), np.sin(grid / 2) ** 2, color="#7b5cfa", lw=2.5, label="formula sin²(θ/2)")
plt.plot(np.degrees(angles), measured, "o", color="#ff8a50", label="500 shots each")
plt.xlabel("θ (degrees)")
plt.ylabel("P(1)")
plt.xticks(range(0, 181, 30))
plt.legend()
plt.show()

## A mystery coin
`mystery_coin` hides its angle. All you get is counts.

In [ ]:
_hidden = np.random.default_rng(2026).uniform(np.radians(20), np.radians(160))

def mystery_coin(shots):
    return sampler.run([coin(_hidden)], shots=shots).result()[0].data.meas.get_counts()

print(mystery_coin(20))

## Project: be the detective
Write `estimate_theta(counts)`. It should return the estimated angle **and** a 95% range for it:
estimate $\hat p$, its error $\sqrt{\hat p(1-\hat p)/N}$, take $\hat p \pm 2$ errors, and turn all three
probabilities into angles with `theta_for`. Then use 2000 shots of the mystery coin.

In [ ]:
def estimate_theta(counts):
    shots = sum(counts.values())
    p = counts.get("1", 0) / shots
    # TODO: the error of p, then the angles for p - 2*error, p and p + 2*error
    return None, (None, None)

estimate, (low, high) = estimate_theta(mystery_coin(2000))
print("estimate:", estimate, " range:", low, high)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def estimate_theta(counts):
    shots = sum(counts.values())
    p = counts.get("1", 0) / shots
    err = np.sqrt(p * (1 - p) / shots)
    low, high = np.clip(p - 2 * err, 0, 1), np.clip(p + 2 * err, 0, 1)
    return np.degrees(theta_for(p)), (np.degrees(theta_for(low)), np.degrees(theta_for(high)))

for shots in [100, 2000, 50000]:
    est, (low, high) = estimate_theta(mystery_coin(shots))
    print(f"{shots:6d} shots: theta = {est:6.2f} degrees, 95% range {low:6.2f} to {high:6.2f}")
print(f"\nThe hidden angle was {np.degrees(_hidden):.2f} degrees.")